## Étape 4.1 : lecture des données

Le CSV est lu depuis HDFS. On vérifie le nombre de colonnes (24 attendues) et de lignes (environ 200 000 attendues).

In [1]:
# Imports et configuration, regroupés une seule fois
import os

import pandas as pd
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.ml.stat import ChiSquareTest, Correlation
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

# Chemin du CSV chargé dans HDFS (modifiable via la variable d'environnement DATA_PATH)
DATA_PATH = os.environ.get(
    "DATA_PATH",
    "hdfs://localhost:9000/user/miguel/projet_3IABD/Bank_Transaction_Fraud_Detection.csv",
)

# Dossier des exports, lu par app.py
OUT = "outputs"
os.makedirs(OUT, exist_ok=True)

# Colonnes utilisées par l'analyse et les modèles
COLONNES_TEXTE = ["genre", "type_compte", "type_transaction", "categorie_marchand", "type_appareil"]
COLONNES_NUM = ["age", "montant", "solde_compte", "heure", "jour_semaine"]

# Seuil au-delà duquel deux variables sont considérées comme redondantes
SEUIL_REDONDANCE = 0.9

# Graine fixe : les découpages et tirages sont reproductibles
SEED = 42

# Session Spark locale, avec tous les cœurs de la machine
spark = SparkSession.builder.appName("fraude").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

26/11/18 23:44:52 WARN Utils: Your hostname, LAPTOP-2O6PEQV6 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/18 23:44:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/18 23:44:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/18 23:44:55 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
# Fonctions utilitaires, réutilisées en 4.3 (exploration) et en 4.5 (sélection sur le train)


def ajuster_encodage(data, drop_last):
    """Ajuste StringIndexer puis OneHotEncoder sur data.

    drop_last=True retire une modalité de référence par colonne, pour éviter la redondance parfaite.
    handleInvalid="keep" garde les catégories inconnues au lieu de faire échouer le calcul.
    """
    indexers = [
        StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
        for c in COLONNES_TEXTE
    ]
    encoder = OneHotEncoder(
        inputCols=[f"{c}_idx" for c in COLONNES_TEXTE],
        outputCols=[f"{c}_oh" for c in COLONNES_TEXTE],
        dropLast=drop_last,
    )
    return Pipeline(stages=indexers + [encoder]).fit(data)


def appliquer_encodage(data, modele, drop_last):
    """Applique l'encodage, puis déplie chaque vecteur one-hot en une colonne 0/1 par modalité.

    Retourne le DataFrame enrichi et la liste des nouvelles colonnes.
    """
    data = modele.transform(data)
    colonnes_oh = []
    for i, c in enumerate(COLONNES_TEXTE):
        modalites = modele.stages[i].labels
        if drop_last:
            # La dernière modalité est la référence : elle n'a pas de colonne
            modalites = modalites[:-1]
        for j, modalite in enumerate(modalites):
            nom = f"{c}_{modalite}".replace(" ", "_")
            data = data.withColumn(nom, vector_to_array(F.col(f"{c}_oh"))[j])
            colonnes_oh.append(nom)
    return data, colonnes_oh


def matrice_correlation(data, colonnes):
    """Matrice de corrélation de Pearson entre les colonnes données, en DataFrame pandas."""
    vecteur = VectorAssembler(inputCols=colonnes, outputCol="vec_corr").transform(data)
    valeurs = Correlation.corr(vecteur, "vec_corr", "pearson").head()[0].toArray()
    return pd.DataFrame(valeurs, index=colonnes, columns=colonnes)


def chi2_et_cramer(data, colonne):
    """Chi², p-value et V de Cramér entre une colonne texte et la cible binaire 'cible'.

    La cible étant binaire, le V de Cramér vaut sqrt(chi² / n).
    """
    indexe = StringIndexer(inputCol=colonne, outputCol="idx_test").fit(data).transform(data)
    vecteur = VectorAssembler(inputCols=["idx_test"], outputCol="v_test").transform(indexe)
    resultat = ChiSquareTest.test(vecteur, "v_test", "cible").head()
    chi2 = float(resultat.statistics[0])
    p_value = float(resultat.pValues[0])
    n = data.count()
    return chi2, p_value, (chi2 / n) ** 0.5

In [3]:
# Lecture depuis HDFS : toutes les colonnes restent en texte, le typage se fait en 4.2
raw = spark.read.csv(DATA_PATH, header=True, inferSchema=False)

print("Nombre de colonnes :", len(raw.columns))
print("Nombre de lignes :", raw.count())

Nombre de colonnes : 24


Nombre de lignes : 200000


## Étape 4.2 : renommage et typage des colonnes

Les colonnes sont renommées en français, puis typées : les montants, l'âge et la cible en nombres, la date en type date, le reste en texte.

In [4]:
# Correspondance entre les noms anglais du fichier (à gauche) et les noms français du notebook (à droite)
RENOMMAGE = {
    "Customer_ID": "id_client",
    "Customer_Name": "nom_client",
    "Gender": "genre",
    "Age": "age",
    "State": "etat",
    "City": "ville",
    "Bank_Branch": "agence_bancaire",
    "Account_Type": "type_compte",
    "Transaction_ID": "id_transaction",
    "Transaction_Date": "date_transaction",
    "Transaction_Time": "heure_transaction",
    "Transaction_Amount": "montant",
    "Merchant_ID": "id_marchand",
    "Transaction_Type": "type_transaction",
    "Merchant_Category": "categorie_marchand",
    "Account_Balance": "solde_compte",
    "Transaction_Device": "appareil_transaction",
    "Transaction_Location": "localisation_transaction",
    "Device_Type": "type_appareil",
    "Is_Fraud": "est_fraude",
    "Transaction_Currency": "devise",
    "Customer_Contact": "contact_client",
    "Transaction_Description": "description_transaction",
    "Customer_Email": "email_client",
}

# Contrôle : toutes les colonnes attendues doivent exister avant toute transformation
manquantes = [c for c in RENOMMAGE if c not in raw.columns]
if manquantes:
    raise ValueError(f"Colonnes absentes du fichier : {manquantes}")

df = raw.select([F.col(c).alias(RENOMMAGE[c]) for c in RENOMMAGE])

df = (
    df
    # Types numériques
    .withColumn("age", F.col("age").cast(IntegerType()))
    .withColumn("est_fraude", F.col("est_fraude").cast(IntegerType()))
    .withColumn("montant", F.col("montant").cast(DoubleType()))
    .withColumn("solde_compte", F.col("solde_compte").cast(DoubleType()))
    # Date au format jj-mm-aaaa, par exemple 23-01-2025
    .withColumn("date_transaction", F.to_date("date_transaction", "dd-MM-yyyy"))
    # Variables dérivées : heure de la journée (0 à 23) et jour de la semaine (1 = dimanche)
    .withColumn("heure", F.hour(F.to_timestamp("heure_transaction", "HH:mm:ss")))
    .withColumn("jour_semaine", F.dayofweek("date_transaction"))
)

# Une valeur qui ne se convertit pas devient null : on compte les nulls pour repérer les problèmes
for c in ["age", "est_fraude", "montant", "solde_compte", "date_transaction", "heure"]:
    print(f"{c} : {df.filter(F.col(c).isNull()).count()} null(s)")

age : 0 null(s)
est_fraude : 0 null(s)
montant : 0 null(s)
solde_compte : 0 null(s)


date_transaction : 0 null(s)


heure : 0 null(s)


## Étape 4.3 : statistiques descriptives

On décrit les colonnes numériques, la répartition de la cible `est_fraude`, puis les liens entre les variables et la fraude. Lors d'une analyse préalable sur le fichier complet, le taux de fraude était d'environ 5 % et les corrélations proches de 0. Ces chiffres devront être retrouvés ici, et interprétés après exécution.

In [5]:
# Jeu complet, avec la cible en nombre décimal (format attendu par Spark ML)
df_t = df.withColumn("cible", F.col("est_fraude").cast("double"))
n = df_t.count()

# 1. Statistiques descriptives et taux de fraude
df_t.select(COLONNES_NUM + ["est_fraude"]).describe().show()
(
    df_t.groupBy("est_fraude").count()
    .withColumn("taux_%", F.round(F.col("count") / n * 100, 2))
    .orderBy("est_fraude")
    .show()
)

# 2. Corrélation de Pearson entre chaque variable numérique et la fraude
for c in COLONNES_NUM:
    print(f"corrélation {c} / est_fraude : {df_t.stat.corr(c, 'cible'):.4f}")

# 3. Chi² et V de Cramér pour chaque colonne texte, sur la colonne d'origine
resultats = []
for c in COLONNES_TEXTE:
    chi2, p_value, v_cramer = chi2_et_cramer(df_t, c)
    resultats.append((c, round(chi2, 2), round(p_value, 6), round(v_cramer, 5)))
tableau_texte = pd.DataFrame(resultats, columns=["colonne", "chi2", "p_value", "v_cramer"])
print(tableau_texte.to_string(index=False))

# 4. Matrice de corrélation : variables numériques, one-hot complets et cible
#    Toutes les modalités sont gardées ici, pour lire les liens entre elles
modele_exploration = ajuster_encodage(df_t, drop_last=False)
df_exploration, colonnes_oh_exploration = appliquer_encodage(df_t, modele_exploration, drop_last=False)
matrice = matrice_correlation(df_exploration, COLONNES_NUM + colonnes_oh_exploration + ["cible"]).round(3)

# Lecture : la colonne cible donne le lien de chaque variable avec la fraude
print(matrice["cible"].sort_values().to_string())

# 5. Exports
df_t.select(COLONNES_NUM + ["est_fraude"]).describe().toPandas().to_csv(f"{OUT}/statistiques.csv", index=False)
tableau_texte.to_csv(f"{OUT}/liens_fraude_texte.csv", index=False)
matrice.to_csv(f"{OUT}/matrice_correlation.csv")
print("Exports écrits dans", OUT)

26/11/18 23:45:15 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+------------------+------------------+------------------+-----------------+-------------------+
|summary|               age|           montant|      solde_compte|             heure|     jour_semaine|         est_fraude|
+-------+------------------+------------------+------------------+------------------+-----------------+-------------------+
|  count|            200000|            200000|            200000|            200000|           200000|             200000|
|   mean|          44.01511|49538.015554200014| 52437.98878355006|          11.49042|         4.037675|            0.05044|
| stddev|15.288774326969515| 28551.87400382295|27399.507128068493|6.9170938590232565|1.942018138830649|0.21885165267877088|
|    min|                18|             10.29|           5000.82|                 0|                1|                  0|
|    max|                70|          98999.98|          99999.95|                23|                7|                  1|
+-------

           colonne  chi2  p_value  v_cramer
             genre  0.08 0.771665   0.00065
       type_compte  3.82 0.148078   0.00437
  type_transaction  4.28 0.369492   0.00463
categorie_marchand  6.82 0.234292   0.00584
     type_appareil  0.68 0.878641   0.00184


categorie_marchand_Entertainment   -0.005
type_compte_Checking               -0.004
type_transaction_Bill_Payment      -0.003
jour_semaine                       -0.003
type_transaction_Withdrawal        -0.003
heure                              -0.002
age                                -0.002
montant                            -0.002
categorie_marchand_Health          -0.001
type_appareil_Mobile               -0.001
genre_Female                       -0.001
type_compte_Savings                -0.000
categorie_marchand_Electronics     -0.000
solde_compte                        0.000
categorie_marchand_Restaurant      -0.000
type_appareil_POS                   0.000
type_appareil_ATM                  -0.000
genre_Male                          0.001
type_transaction_Debit              0.001
type_transaction_Credit             0.001
type_appareil_Desktop               0.002
type_transaction_Transfer           0.003
categorie_marchand_Clothing         0.003
categorie_marchand_Groceries      

## Étape 4.4 : nettoyage

Le nettoyage porte sur cinq points : suppression de la colonne constante `devise`, retrait des données personnelles, doublons sur l'identifiant de transaction, valeurs manquantes, et plages de valeurs. Lors de l'analyse préalable, aucun doublon ni valeur manquante n'a été trouvé : la cellule le vérifie et le documente.

In [6]:
# 1. Colonne constante et données personnelles : retirées de l'analyse
print("Valeurs distinctes de devise :", df.select("devise").distinct().count())
df = df.drop("devise", "nom_client", "contact_client", "email_client")

# 2. Doublons sur l'identifiant de transaction
n_avant = df.count()
df = df.dropDuplicates(["id_transaction"])
print("Doublons supprimés :", n_avant - df.count())

# 3. Valeurs manquantes
n_avant = df.count()
df = df.dropna()
print("Lignes incomplètes supprimées :", n_avant - df.count())

# 4. Plages de valeurs cohérentes : âge entre 18 et 120, montant positif, solde positif ou nul, cible 0 ou 1
n_avant = df.count()
df = df.filter(
    F.col("age").between(18, 120)
    & (F.col("montant") > 0)
    & (F.col("solde_compte") >= 0)
    & F.col("est_fraude").isin(0, 1)
)
print("Lignes incohérentes supprimées :", n_avant - df.count())

print("Lignes après nettoyage :", df.count(), "| colonnes :", len(df.columns))

Valeurs distinctes de devise : 1


Doublons supprimés : 0


Lignes incomplètes supprimées : 0


Lignes incohérentes supprimées : 0


Lignes après nettoyage : 200000 | colonnes : 22


## Étape 4.5 : features

Les modèles ne lisent que des nombres. Cette étape extrait l'heure et le jour de la semaine, encode les colonnes texte, puis regroupe toutes les variables dans un vecteur `features`. Seules les colonnes texte à peu de valeurs distinctes sont retenues ; les colonnes à très nombreuses valeurs (ville, état, agence, localisation, appareil de transaction) sont écartées pour garder un modèle simple.

In [7]:
# 1. Cible en nombre décimal, puis découpage train/test (80 % / 20 %)
df_m = df.withColumn("label", F.col("est_fraude").cast("double"))
train, test = df_m.randomSplit([0.8, 0.2], seed=SEED)
print("Lignes train :", train.count(), "| Lignes test :", test.count())

# 2. Encodage ajusté sur le train, avec une modalité de référence retirée par colonne
modele_train = ajuster_encodage(train, drop_last=True)
train_enc, colonnes_oh = appliquer_encodage(train, modele_train, drop_last=True)
test_enc, _ = appliquer_encodage(test, modele_train, drop_last=True)

# 3. Sélection par redondance, calculée sur le train uniquement
candidates = COLONNES_NUM + colonnes_oh
matrice_train = matrice_correlation(train_enc, candidates)
retenues = []
for c in candidates:
    if any(abs(matrice_train.loc[c, k]) > SEUIL_REDONDANCE for k in retenues):
        print("retirée (redondante) :", c)
    else:
        retenues.append(c)
print("Variables retenues :", len(retenues), "sur", len(candidates))

# 4. Vecteur features : toutes les variables retenues dans une seule colonne
assembleur = VectorAssembler(inputCols=retenues, outputCol="features")
train_feat = assembleur.transform(train_enc).select("id_transaction", "heure", "features", "label")
test_feat = assembleur.transform(test_enc).select("id_transaction", "heure", "features", "label")
train_feat.show(3, truncate=False)

Lignes train : 160365 | Lignes test : 39635


Variables retenues : 20 sur 20


+------------------------------------+-----+----------------------------------------------------------------------------+-----+
|id_transaction                      |heure|features                                                                    |label|
+------------------------------------+-----+----------------------------------------------------------------------------+-----+
|0b280111-41e2-4892-9a82-68b7c9ef8d99|19   |(20,[0,1,2,3,4,5,9,16,17],[68.0,65838.99,53859.94,19.0,5.0,1.0,1.0,1.0,1.0])|0.0  |
|26f49997-13fe-4777-9a26-b7324197433e|21   |(20,[0,1,2,3,4,6,10,13],[32.0,12248.24,95311.65,21.0,1.0,1.0,1.0,1.0])      |0.0  |
|916e4f07-ec38-4b60-8b83-d07636edd9ac|13   |(20,[0,1,2,3,4,5,6],[64.0,90394.8,40615.61,13.0,3.0,1.0,1.0])               |0.0  |
+------------------------------------+-----+----------------------------------------------------------------------------+-----+
only showing top 3 rows



In [8]:
# 1. Poids de classe calculés sur le train uniquement.
#    Toutes les transactions sont gardées : une fraude pèse 1, une transaction normale pèse
#    n_fraude / n_normal, ce qui donne le même poids total aux deux classes.
n_fraude = train_feat.filter(F.col("label") == 1).count()
n_normal = train_feat.filter(F.col("label") == 0).count()
poids_normal = n_fraude / n_normal
train_pond = train_feat.withColumn(
    "poids", F.when(F.col("label") == 1, 1.0).otherwise(poids_normal)
)
print("Poids d'une transaction normale :", round(poids_normal, 4))


# 2. F1 de la classe fraude, calculé à partir de la précision et du rappel
def f1_fraude(d):
    tp = d.filter((F.col("label") == 1) & (F.col("prediction") == 1)).count()
    fp = d.filter((F.col("label") == 0) & (F.col("prediction") == 1)).count()
    fn = d.filter((F.col("label") == 1) & (F.col("prediction") == 0)).count()
    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    rappel = tp / (tp + fn) if tp + fn > 0 else 0.0
    return 2 * precision * rappel / (precision + rappel) if precision + rappel > 0 else 0.0


evaluateur_pr = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderPR")
evaluateur_roc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")


# 3. Matrice de confusion : nombre de transactions par couple (réel, prédit)
def matrice_confusion(pred):
    compte = {
        (int(r["label"]), int(r["prediction"])): r["count"]
        for r in pred.groupBy("label", "prediction").count().collect()
    }
    return pd.DataFrame(
        [
            [compte.get((0, 0), 0), compte.get((0, 1), 0)],
            [compte.get((1, 0), 0), compte.get((1, 1), 0)],
        ],
        index=["réel 0 (normal)", "réel 1 (fraude)"],
        columns=["prédit 0", "prédit 1"],
    )


# 4. Baseline « toujours 0 » : prédit non-fraude pour toutes les transactions du test.
#    Un modèle constant a une AUC-PR égale au taux de fraude du test, et une AUC-ROC de 0,5.
taux_test = test_feat.agg(F.avg("label")).first()[0]
baseline = test_feat.withColumn("prediction", F.lit(0.0))
scores = [{
    "modele": "baseline_toujours_0",
    "f1_fraude": f1_fraude(baseline),
    "aire_pr": taux_test,
    "aire_roc": 0.5,
}]

# 5. Entraînement sur le train pondéré, évaluation sur le test non modifié
modeles = {
    "regression_logistique": LogisticRegression(
        featuresCol="features", labelCol="label", weightCol="poids", maxIter=50
    ),
    "foret_aleatoire": RandomForestClassifier(
        featuresCol="features", labelCol="label", weightCol="poids",
        numTrees=100, maxDepth=8, seed=SEED,
    ),
}
predictions_export = None
for nom, estimateur in modeles.items():
    modele = estimateur.fit(train_pond)
    pred = modele.transform(test_feat)
    scores.append({
        "modele": nom,
        "f1_fraude": f1_fraude(pred),
        "aire_pr": evaluateur_pr.evaluate(pred),
        "aire_roc": evaluateur_roc.evaluate(pred),
    })
    print(f"\n=== Matrice de confusion : {nom} ===")
    print(matrice_confusion(pred))
    if nom == "foret_aleatoire":
        predictions_export = pred

# 6. Tableau des scores
scores_pd = pd.DataFrame(scores).round(4)
print(scores_pd.to_string(index=False))

# 7. Exports pour le dashboard (app.py)
scores_pd.to_csv(f"{OUT}/scores_modeles.csv", index=False)
(
    predictions_export
    .select("id_transaction", F.col("heure").alias("hora_dia"), "label", "prediction")
    .toPandas()
    .to_csv(f"{OUT}/predictions.csv", index=False)
)
print("Exports écrits dans", OUT)

Poids d'une transaction normale : 0.0533


26/11/18 23:46:38 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/11/18 23:46:38 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS



=== Matrice de confusion : regression_logistique ===


                 prédit 0  prédit 1
réel 0 (normal)     18995     18671
réel 1 (fraude)      1011       958


26/11/18 23:47:23 WARN DAGScheduler: Broadcasting large task binary with size 1122.9 KiB
26/11/18 23:47:31 WARN DAGScheduler: Broadcasting large task binary with size 1919.8 KiB
26/11/18 23:47:38 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
26/11/18 23:47:53 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/11/18 23:47:55 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/11/18 23:47:59 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/11/18 23:48:01 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/11/18 23:48:04 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB



=== Matrice de confusion : foret_aleatoire ===


26/11/18 23:48:08 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/11/18 23:48:11 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB


                 prédit 0  prédit 1
réel 0 (normal)     29428      8238
réel 1 (fraude)      1512       457
               modele  f1_fraude  aire_pr  aire_roc
  baseline_toujours_0     0.0000   0.0497    0.5000
regression_logistique     0.0887   0.0486    0.4912
      foret_aleatoire     0.0857   0.0510    0.5128


26/11/18 23:48:13 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB


Exports écrits dans outputs
